Essay evaluator using Parallel Workflow 

In [66]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
import operator
import os

In [67]:
load_dotenv()

llm = ChatOpenAI()

In [68]:
class FeedbackSchema(BaseModel):
    feedback:str = Field(description='Eassy evaluation feedback provided by the LLM')
    score: int = Field(description='Eassy evaluation score provided by the LLM')

structured_output_llm = llm.with_structured_output(FeedbackSchema)

In [69]:
#define State
class EssayEvaluationState(TypedDict):
    essay_text:str
    feedback_language:str
    feedback_analysis:str
    feedback_thought:str
    individual_scores: Annotated[list[int], operator.add]
    feedback_summary:str
    final_score:float

In [70]:
def evaluate_language(state:EssayEvaluationState):
    """This function evaluates the essay based on language"""
    
    prompt=f"""
        you are a essay evaluator. Evaluate the essay {state['essay_text']} based on language and provide a feedback and score
    """
    result = structured_output_llm.invoke(prompt)
    return {'feedback_language': result.feedback, 'individual_scores': [result.score]}

def evaluate_analysis(state:EssayEvaluationState):
    """This function evaluates the essay based on analysis"""
    
    prompt=f"""
        you are a essay evaluator. Evaluate the essay {state['essay_text']} based on analysis and provide a feedback and score
    """
    result = structured_output_llm.invoke(prompt)
    return {'feedback_analysis': result.feedback ,'individual_scores': [result.score]}


def evaluate_thought(state:EssayEvaluationState):
    """This function evaluates the essay based on thought"""
    
    prompt=f"""
        you are a essay evaluator. Evaluate the essay {state['essay_text']} based on thought and provide a feedback and score
    """
    result = structured_output_llm.invoke(prompt)
    return {'feedback_thought': result.feedback, 'individual_scores': [result.score]}


def summarize_feedback(state:EssayEvaluationState):
    """This function summatrizes the feedback and generate a summarized feedback along with a final scorre"""
    
    prompt=f"""
        you are a summarizer. Summarize the feedback based on individual feedbacks {state['feedback_language']} , {state['feedback_analysis']}, {state['feedback_thought']} 
    """
    result = llm.invoke(prompt)
    
    final_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {'feedback_summary': result.content, 'final_score': final_score}


In [71]:
graph = StateGraph(EssayEvaluationState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('summarize_feedback',summarize_feedback)

graph.add_edge(START ,'evaluate_language')
graph.add_edge(START ,'evaluate_analysis')
graph.add_edge(START ,'evaluate_thought')


graph.add_edge('evaluate_language' ,'summarize_feedback')
graph.add_edge('evaluate_analysis' ,'summarize_feedback')
graph.add_edge('evaluate_thought' ,'summarize_feedback')

graph.add_edge('summarize_feedback' ,END)


workflow = graph.compile()




In [72]:
initial_state = {'essay_text':"""The Role of Technology in Modern Society

Technology has become an inseparable part of modern life, shaping the way people communicate, work, and think. Over the past few decades, rapid advancements in digital tools and systems have transformed societies across the globe. From smartphones to artificial intelligence, technology continues to redefine human potential and the structure of everyday life.

One of the most significant impacts of technology is in communication. The rise of the internet and mobile devices has made it possible for individuals to connect instantly, regardless of geographical boundaries. Social media platforms, messaging apps, and video conferencing tools have brought people closer, enabling real-time interaction and collaboration. This has not only strengthened personal relationships but also enhanced global business operations and cultural exchange.

In the field of education, technology has opened new avenues for learning. Online courses, virtual classrooms, and digital resources allow students to access information anytime and anywhere. This flexibility has made education more inclusive, reaching individuals who may not have access to traditional institutions. Moreover, interactive tools and multimedia content have improved the overall learning experience, making it more engaging and effective.

However, the widespread use of technology also presents challenges. Issues such as data privacy, cybersecurity threats, and digital addiction are becoming increasingly prevalent. Over-reliance on technology can reduce face-to-face interactions and impact mental health. Therefore, it is essential to strike a balance between utilizing technology and maintaining human connections.

In conclusion, technology plays a crucial role in modern society by enhancing communication, education, and productivity. While it offers numerous benefits, it also requires responsible usage and ethical considerations. As technology continues to evolve, individuals and societies must adapt wisely to ensure that its impact remains positive and sustainable.

If you tell"""}
output_state = workflow.invoke(initial_state)
print(output_state)


{'essay_text': 'The Role of Technology in Modern Society\n\nTechnology has become an inseparable part of modern life, shaping the way people communicate, work, and think. Over the past few decades, rapid advancements in digital tools and systems have transformed societies across the globe. From smartphones to artificial intelligence, technology continues to redefine human potential and the structure of everyday life.\n\nOne of the most significant impacts of technology is in communication. The rise of the internet and mobile devices has made it possible for individuals to connect instantly, regardless of geographical boundaries. Social media platforms, messaging apps, and video conferencing tools have brought people closer, enabling real-time interaction and collaboration. This has not only strengthened personal relationships but also enhanced global business operations and cultural exchange.\n\nIn the field of education, technology has opened new avenues for learning. Online courses, 